# Getting experiment shear stresses 

In [18]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# import topography data
topo = pd.read_csv('XS_topo/experiment_reach.csv')
# import depth data and make date the index
exp2 = pd.read_csv('water_depth/exp_depths/exp2.csv', index_col='Date_Time', parse_dates=True)
exp3 = pd.read_csv('water_depth/exp_depths/exp3.csv', index_col='Date_Time', parse_dates=True)
exp4 = pd.read_csv('water_depth/exp_depths/exp4.csv', index_col='Date_Time', parse_dates=True)
exp5 = pd.read_csv('water_depth/exp_depths/exp5.csv', index_col='Date_Time', parse_dates=True)
exp6 = pd.read_csv('water_depth/exp_depths/exp6.csv', index_col='Date_Time', parse_dates=True)
exp7 = pd.read_csv('water_depth/exp_depths/exp7.csv', index_col='Date_Time', parse_dates=True)
exp8 = pd.read_csv('water_depth/exp_depths/exp8.csv', index_col='Date_Time', parse_dates=True)
# get rid of Nan values
exp2 = exp2.dropna()
exp3 = exp3.dropna()
exp4 = exp4.dropna()
exp5 = exp5.dropna()
exp6 = exp6.dropna()
exp7 = exp7.dropna()
exp8 = exp8.dropna()


C:\Users\huck4481\AppData\Local\Temp\ipykernel_21976\4037076757.py:9: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  exp2 = pd.read_csv('water_depth/exp_depths/exp2.csv', index_col='Date_Time', parse_dates=True)
C:\Users\huck4481\AppData\Local\Temp\ipykernel_21976\4037076757.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  exp3 = pd.read_csv('water_depth/exp_depths/exp3.csv', index_col='Date_Time', parse_dates=True)
C:\Users\huck4481\AppData\Local\Temp\ipykernel_21976\4037076757.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  exp4 = pd.read_csv('water_depth/exp_dept

In [19]:
topo

,distance,elevation
0,0.0,2756.000
1,0.1,2755.979
2,0.2,2756.001
3,0.3,2755.922
4,0.4,2755.880
5,0.5,2755.793
6,0.6,2755.740
7,0.7,2755.650
8,0.8,2755.623
9,0.9,2755.628


### Function Definition - Hydraulic Radius

In [20]:
def get_hydraulic_radius(X, Y, WSE):
    wet_areas = []
    wet_perimeters = []
    poly_x = []
    poly_y = []

    # Loop through the points of the cross-section to find wet polygon
    for i in range(len(X) - 1):
        y1, y2 = Y[i], Y[i+1]
        x1, x2 = X[i], X[i+1]

        # First, checking if we enter the water
        if (y1 >= WSE and y2 < WSE):
            # Linear interpolation to find the intersection point
            x_int1 = x1 + (WSE - y1) * (x2 - x1) / (y2 - y1)
            poly_x.append(x_int1)
            poly_y.append(WSE)
        
        # If not, we check if the first point is below the WSE
        if (y1 < WSE):
            poly_x.append(x1)
            poly_y.append(y1)
        
        # Then, we check if we go out of the water. If we did, we calculate the area and store it
        if (y1 < WSE and y2 >= WSE):
            # Linear interpolation to find the intersection point
            x_int2 = x1 + (WSE - y1) * (x2 - x1) / (y2 - y1)
            poly_x.append(x_int2)
            poly_y.append(WSE)

            # Closing the polygon
            if len(poly_x) > 0:
                poly_x.append(poly_x[0])
                poly_y.append(poly_y[0])
            
            # Calculating the area
            tempCalc1 = 0.5 * np.abs(np.dot(poly_x, np.roll(poly_y, 1)) - np.dot(poly_y, np.roll(poly_x, 1)))
            wet_areas.append(tempCalc1)

            # Calculating the wet perimeter
            distances = np.sqrt(np.diff(poly_x)**2 + np.diff(poly_y)**2)
            wet_perimeters.append(np.sum(distances[:-1]))

            # Resetting the polygon
            poly_x = []
            poly_y = []
    
    # return hydraulic radius, wet area, wet perimeter
    tempCalc1 = np.sum(wet_areas)
    tempCalc2 = np.sum(wet_perimeters)
    # check if tempCalc2 is zero to avoid division by zero
    if tempCalc2 == 0:
        hydraulic_radius = float('nan')  
    else:
        hydraulic_radius = tempCalc1 / tempCalc2
    
    return hydraulic_radius

def calculate_hydraulic_radius(topography_df, depth_timeseries, depth_location):
    X = topography_df['distance'].values
    Z = topography_df['elevation'].values
    depth = depth_timeseries['depth'].values
    times = depth_timeseries.index 

    #find the riverbed elevation at the depth measurement location
    riverbed_elevation_at_depth_location = np.interp(depth_location, X, Z)

    hyd_rad = [] 

    # loop over each time step and calculate hydraulic radius
    for d in depth:
        wse = riverbed_elevation_at_depth_location + d  # wse at this time step
        hydraulic_radius_calc = get_hydraulic_radius(X, Z, wse)  # calculate hydraulic radius
        hyd_rad.append(hydraulic_radius_calc)

    # create a DataFrame with time and hydraulic radius for each time step
    average_depth_timeseries = pd.DataFrame({
        'time': times,
        'average_depth': hyd_rad
    }).set_index('time')

    return average_depth_timeseries

### Obtaining hydraulic radius by cross section

In [21]:
exp2_hyd_rad = calculate_hydraulic_radius(topo, exp2, 1.3)
exp3_hyd_rad = calculate_hydraulic_radius(topo, exp3, 1.3)
exp4_hyd_rad = calculate_hydraulic_radius(topo, exp4, 1.3)
exp5_hyd_rad = calculate_hydraulic_radius(topo, exp5, 1.3)
exp6_hyd_rad = calculate_hydraulic_radius(topo, exp6, 1.3)
exp7_hyd_rad = calculate_hydraulic_radius(topo, exp7, 1.3)
exp8_hyd_rad = calculate_hydraulic_radius(topo, exp8, 1.3)

In [22]:
exp2_hyd_rad

,average_depth
time,
2026-07-23 11:57:09,0.046365
2026-07-23 11:57:21,0.084464
2026-07-23 11:57:32,0.077274
2026-07-23 11:57:42,0.066625
2026-07-23 11:57:53,0.058528
2026-07-23 11:58:04,0.054310
2026-07-23 11:58:14,0.051591
2026-07-23 11:58:25,0.051200
2026-07-23 11:58:34,0.051200


### Calculating shear stress as:

$\tau = g \rho s R$

In [23]:
# assigning the other variables
rho = 1000  # density of water in kg/m^3
g = 9.81  # acceleration due to gravity in m/s^2
s = 0.081 # reach slope in m/m

# calculate shear stress for each time step
exp2_shear_stress = rho * g * s * exp2_hyd_rad
exp3_shear_stress = rho * g * s * exp3_hyd_rad
exp4_shear_stress = rho * g * s * exp4_hyd_rad
exp5_shear_stress = rho * g * s * exp5_hyd_rad
exp6_shear_stress = rho * g * s * exp6_hyd_rad
exp7_shear_stress = rho * g * s * exp7_hyd_rad
exp8_shear_stress = rho * g * s * exp8_hyd_rad

In [24]:
# export the shear stress data to CSV files
exp2_shear_stress.to_csv('experiment_shear_stress/exp2_shear_stress.csv')
exp3_shear_stress.to_csv('experiment_shear_stress/exp3_shear_stress.csv')
exp4_shear_stress.to_csv('experiment_shear_stress/exp4_shear_stress.csv')
exp5_shear_stress.to_csv('experiment_shear_stress/exp5_shear_stress.csv')
exp6_shear_stress.to_csv('experiment_shear_stress/exp6_shear_stress.csv')
exp7_shear_stress.to_csv('experiment_shear_stress/exp7_shear_stress.csv')
exp8_shear_stress.to_csv('experiment_shear_stress/exp8_shear_stress.csv')